In [1]:
from AeroTransformerPredict import AeroTransformer, integrate_wing_aerodynamics
import torch
import numpy as np
import json
from datasetProcess import WingDataset, wing_collate_fn
from torch.utils.data import random_split
from wingVAE import WingVAE
import os
from math import atan2, degrees
from WingScriptGenerator import param_to_ggs_wing_dict, calculate_wing_mac
from pathlib import Path
import pandas as pd

In [2]:
aero_transformer_predict = AeroTransformer()

🚀 加载AeroTransformer模型...
loading model...
loading single model ATsurf_L_v1...


c:\Users\WednZ\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
d:\generate_3\aerotransformer_project\flogen\flowvae\ml_operator\operator.py:112: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary obj

In [3]:
full_dataset = WingDataset("D:/generate_3/Wing_generation/data/Train_data/train_dataV7.pt")
dataset_size = len(full_dataset)
train_size = int(0.8 * dataset_size)
val_size = int(0.1 * dataset_size)
test_size = dataset_size - train_size - val_size  # 剩余部分给测试集
# 设置 generator_ 保证每次划分结果一致
_, _, test_dataset = random_split(
    full_dataset, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(62) 
)


d:\generate_3\Wing_generation\src\datasetProcess.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data = torch.load(data_path)


In [4]:
def find_nearest_two_numbers(lst, target):
    nearest1 = None
    nearest2 = None
    min_diff1 = float('inf')
    min_diff2 = float('inf')
    pos1 = None
    pos2 = None
    for index, num in enumerate(lst):
        diff = abs(num - target)
        if num < target:
            if diff < min_diff1:
                min_diff2 = min_diff1
                nearest2 = nearest1
                pos2 = pos1
                min_diff1 = diff
                nearest1 = num
                pos1 = index
            elif diff < min_diff2 and num!=nearest1:
                min_diff2 = diff
                nearest2 = num
                pos2 = index
        else:
            if diff < min_diff2:
                min_diff2 = diff
                nearest2 = num
                pos2 = index
            elif diff < min_diff1 and num!=nearest2:
                min_diff1 = diff
                nearest1 = num
                pos1 = index
    return nearest1, pos1, nearest2, pos2

In [5]:
def calculate_aerodynamic_error(coeff_target:np.ndarray, coeff_pred:np.ndarray):
    pred_cl = coeff_pred[:, 0]
    pred_cd = coeff_pred[:, 1]
    pred_cm = coeff_pred[:, 2]
    
    target_cl = coeff_target[:, 0]
    target_cd = coeff_target[:, 1]
    target_cm = coeff_target[:, 2]

    total_abs_error_cd = 0
    total_abs_error_cm = 0
    total_rlt_error_cd = 0
    total_rlt_error_cm = 0
    numbers = 0
    
    for cl_index in range(len(target_cl)):
        local_cl = target_cl[cl_index]
        local_cd = target_cd[cl_index]
        local_cm = target_cm[cl_index]
        cl_1,pos_1,cl_2,pos_2 = find_nearest_two_numbers(pred_cl, local_cl)
        abs_error_cd = local_cd-pred_cd[pos_1]-(pred_cd[pos_2]-pred_cd[pos_1])/(cl_2-cl_1)*(local_cl-cl_1)
        abs_error_cm = local_cm-pred_cm[pos_1]-(pred_cm[pos_2]-pred_cm[pos_1])/(cl_2-cl_1)*(local_cl-cl_1)

        rlt_error_cd = abs_error_cd/abs(local_cd)
        rlt_error_cm = abs_error_cm/abs(local_cm)

        total_abs_error_cd += abs_error_cd
        total_abs_error_cm += abs_error_cm
        total_rlt_error_cd += rlt_error_cd
        total_rlt_error_cm += rlt_error_cm
        numbers += 1
    
    return total_abs_error_cd/numbers, total_abs_error_cm/numbers, total_rlt_error_cd/numbers, total_rlt_error_cm/numbers #cd_error, cm_error, rlt_cd_error, rlt_cm_error


def cal_ac_loss(surrogate_model, stl_path, root_chord, area, mac, ac_target):
    '''
    输入是目标气动力指标，从指标中提取出评价工况，在马赫数一致，攻角范围进行放大的情况下获得对应地CL，CD，CM
    绘制出极曲线图和俯仰力矩曲线图，计算升力系数一定时，目标到曲线的豪斯多夫距离
    '''
    cos_val = ac_target[:, 1]
    sin_val = ac_target[:, 0]
    mach_val = ac_target[:, 2]
    ac_target = np.array(ac_target)

    # 使用atan2计算角度（弧度）
    angle = torch.rad2deg(torch.atan2(sin_val, cos_val))

    # 构造新的tensor：第0列是角度值，第1列是mach数
    working_condition = torch.stack([angle, mach_val], dim=1)
    # print(working_condition)

    #aero transformer要求对每个几何建一张图
    surrogate_model.build_grid(stl_path, root_chord=root_chord, area=area, mac=mac)
    
    # 每个工况调用一次代理模型，获得一组对应的CL，CD，CM
    cd_error_list = []
    cm_error_list = []
    rlt_cd_error_list = []
    rlt_cm_error_list = []

    for wc_i, wc in enumerate(working_condition):
        # 构造输入序列，先取出攻角，在其±2°的范围内进行采样，步长0.1，然后得到相同长度的马赫数序列
        angle_range = torch.linspace(wc[0] - 2, wc[0] + 2, 40)
        mach_range = torch.full_like(angle_range, wc[1])
        
        coeff_target = ac_target[wc_i, 3:6].reshape(-1, 3)

        # 调用代理模型
        forces = surrogate_model.predict_coeffs(angle_range, mach_range)
        # forces_np = forces.numpy()
        coeff_pred = forces[:, :3]

        # 计算气动力指标的误差
        cd_error, cm_error, rlt_cd_error, rlt_cm_error = calculate_aerodynamic_error(coeff_target, coeff_pred)
        print(f"工况{wc_i}: cd_error={cd_error:.4f}, cm_error={cm_error:.4f}, rlt_cd_error={rlt_cd_error:.4f}, rlt_cm_error={rlt_cm_error:.4f}")
        cd_error_list.append(cd_error)
        cm_error_list.append(cm_error)
        rlt_cd_error_list.append(rlt_cd_error)
        rlt_cm_error_list.append(rlt_cm_error)
    
    return cd_error_list, cm_error_list, rlt_cd_error_list, rlt_cm_error_list, working_condition

        

In [6]:
def cal_ac_hybrid_loss(surrogate_model, stl_path, root_chord, area, mac, ac_target):
    '''
    混合气动力损失计算函数：同时评估代理模型直接预测与物理积分预测。
    
    1. 保留 surrogate_model.predict_coeffs 获取代理模型预测曲线
    2. 逐点调用 aero_transformer_predict + integrate_wing_aerodynamics 获取物理积分曲线
    3. 分别对两条曲线调用 calculate_aerodynamic_error 计算损失
    4. 返回两组损失列表、两组预测系数曲线以及工况信息
    '''
    # ===================== 1. 解析目标工况 =====================
    cos_val = ac_target[:, 1]
    sin_val = ac_target[:, 0]
    mach_val = ac_target[:, 2]
    ac_target_np = np.array(ac_target)

    angle = torch.rad2deg(torch.atan2(sin_val, cos_val))
    working_condition = torch.stack([angle, mach_val], dim=1)

    # 构建几何网格
    surrogate_model.build_grid(stl_path, root_chord=root_chord, area=area, mac=mac)

    # 预计算积分参考参数
    s_ref = area / (root_chord ** 2)
    c_ref = mac / root_chord

    # ===================== 2. 初始化返回值容器 =====================
    # 代理模型(Surrogate)相关
    surr_cd_err, surr_cm_err = [], []
    surr_rlt_cd_err, surr_rlt_cm_err = [], []
    surr_coeff_pred_list = []

    # 物理积分(Integral)相关
    intg_cd_err, intg_cm_err = [], []
    intg_rlt_cd_err, intg_rlt_cm_err = [], []
    intg_coeff_pred_list = []

    # ===================== 3. 逐工况双路径计算 =====================
    for wc_i, wc in enumerate(working_condition):
        # 构造 ±2° 攻角采样序列 (40个点)
        angle_range = torch.linspace(wc[0] - 2, wc[0] + 2, 40)
        mach_range = torch.full_like(angle_range, wc[1])
        coeff_target = ac_target_np[wc_i, 3:6].reshape(-1, 3)

        # ---------- 路径A: 代理模型批量预测 ----------
        forces = surrogate_model.predict_coeffs(angle_range, mach_range)
        surr_coeff_pred = forces[:, :3]  # (40, 3) [CL, CD, CM]

        # ---------- 路径B: 物理积分逐点预测 ----------
        integral_coeffs_list = []
        for pt_idx in range(len(angle_range)):
            flowfield, origeom, _ = aero_transformer_predict.predict_flow_field(
                aoa=angle_range[pt_idx], mach=mach_range[pt_idx]
            )
            inp = torch.tensor(origeom, dtype=torch.float32).unsqueeze(0)
            outputs = torch.tensor(flowfield, dtype=torch.float32)
            res = integrate_wing_aerodynamics(
                inp, outputs,
                s_ref=s_ref, c_ref=c_ref,
                x_ref=0, y_ref=0,
                alpha_deg=angle_range[pt_idx],
                include_friction=False
            )
            integral_coeffs_list.append([res['CL'], res['CD'], res['CM']])

        intg_coeff_pred = torch.tensor(integral_coeffs_list, dtype=torch.float32)  # (40, 3)

        # ---------- 分别计算两组误差 ----------
        s_cd, s_cm, s_rcd, s_rcm = calculate_aerodynamic_error(coeff_target, surr_coeff_pred)
        i_cd, i_cm, i_rcd, i_rcm = calculate_aerodynamic_error(coeff_target, intg_coeff_pred)

        # ---------- 记录结果 ----------
        surr_cd_err.append(s_cd);   surr_cm_err.append(s_cm)
        surr_rlt_cd_err.append(s_rcd); surr_rlt_cm_err.append(s_rcm)
        surr_coeff_pred_list.append(surr_coeff_pred)

        intg_cd_err.append(i_cd.item());   intg_cm_err.append(i_cm.item())
        intg_rlt_cd_err.append(i_rcd.item()); intg_rlt_cm_err.append(i_rcm.item())
        intg_coeff_pred_list.append(intg_coeff_pred.numpy())

        print(f"工况{wc_i}的loss | Surrogate: cd={s_cd:.4f} cm={s_cm:.4f} | "
              f"Integral: cd={i_cd:.4f} cm={i_cm:.4f}")

    return {
        'surrogate': {
            'cd_error': surr_cd_err, 'cm_error': surr_cm_err,
            'rlt_cd_error': surr_rlt_cd_err, 'rlt_cm_error': surr_rlt_cm_err,
            'coeff_pred': surr_coeff_pred_list   # List[np.ndarray], shape=(N_wc, 40, 3)
        },
        'integral': {
            'cd_error': intg_cd_err, 'cm_error': intg_cm_err,
            'rlt_cd_error': intg_rlt_cd_err, 'rlt_cm_error': intg_rlt_cm_err,
            'coeff_pred': intg_coeff_pred_list   # List[np.ndarray], shape=(N_wc, 40, 3)
        },
        'working_condition': working_condition
    }

In [7]:
WORKING_DIR = Path('D:/generate_3/Wing_generation/test_files_1')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
# 建立一张总表保存完整信息
test_df = pd.DataFrame(columns=['id', 
                                'aoa', 
                                'mach',
                                'target_ar', 
                                'target_cl', 
                                'target_cd', 
                                'target_cm', 
                                'generated_ar', 
                                'cd_loss_surr',
                                'cm_loss_surr',
                                'cd_loss_intg',
                                'cm_loss_intg'
                                ]) 

for i, item in enumerate(test_dataset):
    # item 是原始数据集的一个样本
    # print(f"Sample {i}: {item.shape if hasattr(item, 'shape') else item}")
    conditions = item["conditions"]
    # print(conditions)
    
    ar_target = conditions["ar_target"]
    ac_target = conditions["ac"]
    # print(ar_target.shape, ac_target.shape) 
    

    # 构造输入
    cond_dict = {
        'ar_target': ar_target.unsqueeze(0),
        'c_ref' : torch.tensor(float(0.3)),
        'ac' : ac_target.unsqueeze(0),
    }
    '''
    用整体的约束作为输入，得到一个几何。然后调用AeroTransformer进行预测。
    每个工况固定马赫数，对攻角进行缩放，然后比较目标系数和预测系数的豪斯多夫距离
    '''
    with open(WORKING_DIR / f'wing_{i}.json', 'r') as f:
        ggs_wing_dict = json.load(f)
    geo_char = calculate_wing_mac(ggs_wing_dict)
    print(geo_char)

    # 计算AR
    generated_ar = geo_char['span']**2 / geo_char['area'] *2
    print(generated_ar, ar_target)
    geo_loss_rel = (generated_ar - ar_target.item()) / ar_target.item()
    geo_loss_abs = generated_ar - ar_target.item()

    # 计算AC损失
    stl_path = WORKING_DIR / f'wing_{i}.stl'
    root_chord = ggs_wing_dict['init']['chord_root']
    area = geo_char['area']
    mac = geo_char['mac_length']
    
    ret_dict = cal_ac_hybrid_loss(aero_transformer_predict, stl_path, root_chord, area, mac, ac_target)

    #写入数据
    for j in range(len(ret_dict['working_condition'])):
        new_row = {
            'id': i,
            'aoa': ret_dict['working_condition'][j, 0].item(),
            'mach': ret_dict['working_condition'][j, 1].item(),
            'target_ar': ar_target.item(),
            'target_cl': ac_target[j, 3].item() if ac_target.shape[1] > 3 else None,
            'target_cd': ac_target[j, 4].item() if ac_target.shape[1] > 4 else None,
            'target_cm': ac_target[j, 5].item() if ac_target.shape[1] > 5 else None,
            'generated_ar': generated_ar,
            'cd_loss_surr':ret_dict['surrogate']['cd_error'][j],
            'cm_loss_surr':ret_dict['surrogate']['cm_error'][j],
            'cd_loss_intg':ret_dict['integral']['cd_error'][j],
            'cm_loss_intg':ret_dict['integral']['cm_error'][j]
        }
        test_df = pd.concat([test_df, pd.DataFrame([new_row])], ignore_index=True)

{'area': 0.833671561275237, 'mac_length': 0.34271012679189067, 'x_lemac': 0.5637552431041725, 'span': 2.871089910181204}
19.77555108089583 tensor([18.6998])
📂 加载STL文件: D:\generate_3\Wing_generation\test_files_1\wing_0.stl
⏳ 开始进行展向站位切片插值（257x129节点阵列）...
✅ 节点网格(Node)生成成功。最终矩阵形状: (257, 129, 3)
🚀 中心点网格(Cell)计算完毕！最终矩阵形状: (256, 128, 3)
工况0的loss | Surrogate: cd=0.0060 cm=-0.0383 | Integral: cd=0.0112 cm=-0.0116
工况1的loss | Surrogate: cd=0.0056 cm=-0.1058 | Integral: cd=0.0105 cm=-0.0847
工况2的loss | Surrogate: cd=0.0017 cm=0.3252 | Integral: cd=-0.0410 cm=-0.1282
工况3的loss | Surrogate: cd=0.0075 cm=-0.4392 | Integral: cd=0.0090 cm=-0.4322
工况4的loss | Surrogate: cd=-0.0455 cm=-0.9023 | Integral: cd=-0.0234 cm=-0.9741
工况5的loss | Surrogate: cd=0.0036 cm=-0.5381 | Integral: cd=0.0073 cm=-0.5286
工况6的loss | Surrogate: cd=0.0001 cm=0.3290 | Integral: cd=-0.0102 cm=0.2681
工况7的loss | Surrogate: cd=0.0059 cm=-0.1662 | Integral: cd=0.0101 cm=-0.1492
{'area': 0.7034170317372759, 'mac_length': 0.35962127881941

C:\Users\WednZ\AppData\Local\Temp\ipykernel_4708\484541906.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  test_df = pd.concat([test_df, pd.DataFrame([new_row])], ignore_index=True)


✅ 节点网格(Node)生成成功。最终矩阵形状: (257, 129, 3)
🚀 中心点网格(Cell)计算完毕！最终矩阵形状: (256, 128, 3)
工况0的loss | Surrogate: cd=0.0006 cm=-0.1438 | Integral: cd=0.0045 cm=-0.1140
工况1的loss | Surrogate: cd=-0.0507 cm=-0.6858 | Integral: cd=0.0026 cm=-0.8379
工况2的loss | Surrogate: cd=0.0047 cm=-0.3916 | Integral: cd=0.0063 cm=-0.3769
工况3的loss | Surrogate: cd=-0.0896 cm=-0.6437 | Integral: cd=-0.0151 cm=-0.8603
工况4的loss | Surrogate: cd=-0.1140 cm=-0.6206 | Integral: cd=-0.0229 cm=-0.8605
工况5的loss | Surrogate: cd=-0.0040 cm=0.1209 | Integral: cd=0.0028 cm=0.1562
工况6的loss | Surrogate: cd=-0.0509 cm=-0.6774 | Integral: cd=-0.0012 cm=-0.8276
工况7的loss | Surrogate: cd=0.0009 cm=-0.1620 | Integral: cd=0.0046 cm=-0.1327
{'area': 0.4969307961948514, 'mac_length': 0.430022134120276, 'x_lemac': 0.4616351750621483, 'span': 1.6675640115139538}
11.19177862909564 tensor([12.0675])
📂 加载STL文件: D:\generate_3\Wing_generation\test_files_1\wing_2.stl
⏳ 开始进行展向站位切片插值（257x129节点阵列）...
✅ 节点网格(Node)生成成功。最终矩阵形状: (257, 129, 3)
🚀 中心点网格(Cell)计算

d:\generate_3\Wing_generation\src\AeroTransformerPredict.py:43: UserWarning: ⚠️ 输入的 STL 模型非严格闭合，切片阶段可能会产生零散空集。
  warnings.warn("⚠️ 输入的 STL 模型非严格闭合，切片阶段可能会产生零散空集。")


✅ 节点网格(Node)生成成功。最终矩阵形状: (257, 129, 3)
🚀 中心点网格(Cell)计算完毕！最终矩阵形状: (256, 128, 3)
工况0的loss | Surrogate: cd=0.0072 cm=-0.4517 | Integral: cd=0.0062 cm=-0.4432
工况1的loss | Surrogate: cd=-1.7198 cm=3.7696 | Integral: cd=-0.5963 cm=0.2632
工况2的loss | Surrogate: cd=0.1616 cm=-1.0354 | Integral: cd=0.0029 cm=-0.6125
工况3的loss | Surrogate: cd=0.1878 cm=-1.0719 | Integral: cd=0.0033 cm=-0.6218
工况4的loss | Surrogate: cd=0.3115 cm=-1.4202 | Integral: cd=0.0001 cm=-0.6486
工况5的loss | Surrogate: cd=-0.5810 cm=0.8277 | Integral: cd=-0.0013 cm=-0.6584
工况6的loss | Surrogate: cd=0.0047 cm=-0.1264 | Integral: cd=0.0063 cm=-0.1156
工况7的loss | Surrogate: cd=2.4542 cm=-7.0931 | Integral: cd=-0.0776 cm=-0.5536
{'area': 0.6704255860959705, 'mac_length': 0.3521919357838747, 'x_lemac': 0.4316034702095767, 'span': 2.2655691897108174}
15.312075970299189 tensor([15.2524])
📂 加载STL文件: D:\generate_3\Wing_generation\test_files_1\wing_35.stl
⏳ 开始进行展向站位切片插值（257x129节点阵列）...
✅ 节点网格(Node)生成成功。最终矩阵形状: (257, 129, 3)
🚀 中心点网格(Cell)计算完

In [8]:
test_df

,id,aoa,mach,target_ar,target_cl,target_cd,target_cm,generated_ar,cd_loss_surr,cm_loss_surr,cd_loss_intg,cm_loss_intg
0,0,4.204316,0.892057,18.699774,0.024250,0.030144,0.009429,19.775551,0.005982,-0.038289,0.011151,-0.011620
1,0,4.263801,0.886119,18.699774,0.074152,0.026328,0.065842,19.775551,0.005566,-0.105813,0.010501,-0.084705
2,0,2.565042,0.895851,18.699774,-0.195043,0.046496,-0.180569,19.775551,0.001672,0.325207,-0.040995,-0.128234
3,0,5.921206,0.893555,18.699774,0.306286,0.031256,0.278402,19.775551,0.007468,-0.439169,0.008969,-0.432154
4,0,9.438770,0.892127,18.699774,0.716546,0.082097,0.646676,19.775551,-0.045451,-0.902309,-0.023368,-0.974077
...,...,...,...,...,...,...,...,...,...,...,...,...
2889,424,3.737263,0.774208,17.574440,0.045238,0.009051,0.063800,18.072464,0.000246,0.021610,0.005662,0.027738
2890,424,9.620502,0.756635,17.574440,0.761764,0.029984,0.678406,18.072464,-0.938833,0.308266,-0.052147,-0.890784
2891,424,11.256461,0.767318,17.574440,0.954164,0.061178,0.842414,18.072464,-4.160348,4.903195,-1.767765,0.935509
2892,424,7.025143,0.750949,17.574440,0.432807,0.014998,0.398883,18.072464,0.004220,-0.496073,0.004965,-0.479700


In [10]:
test_df.to_csv('D:/generate_3/Wing_generation/exp/Transformer_exp25/data_5.csv', index=True)